In [1]:
import polars as pl
import os
from pathlib import Path

# Polars 출력 설정 - 모든 행과 컬럼 표시
pl.Config.set_tbl_rows(999)
pl.Config.set_tbl_cols(999)
pl.Config.set_tbl_width_chars(120)

# CSV 파일의 절대경로
notebook_dir = Path('.').resolve()
csv_path = notebook_dir.parent / 'output' / 'all_trials_timeseries.csv'

print(f"Current dir: {notebook_dir}")
print(f"CSV Path: {csv_path}")
print(f"Exists: {csv_path.exists()}")

# CSV 읽기
df = pl.read_csv(csv_path)

print("\n" + "="*80)
print("STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)")
print("="*80)

# Trial별 step_TF 결정 (각 trial은 하나의 step_TF 값을 가짐)
trial_level = df.select(['subject', 'velocity', 'trial', 'step_TF']).unique().sort(
    ['subject', 'velocity', 'trial']
)

print("\n[1] Trial별 Step/Nonstep 분포")
print("-" * 50)
trial_step_dist = trial_level.group_by('step_TF').agg(
    pl.len().alias('trial_count')
).with_columns(
    ratio_percent = (pl.col('trial_count') / pl.col('trial_count').sum() * 100).round(2)
).sort('step_TF', descending=True)
print(trial_step_dist)

print("\n[2] 피험자별 Step/Nonstep Trial 개수")
print("-" * 50)
subject_step_count = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count'),
    pl.col('velocity').n_unique().alias('velocity_type_count')
).sort(['subject', 'step_TF'])
print(subject_step_count)

print("\n[3] 피험자별 총 Trial 개수")
print("-" * 50)
subject_total = trial_level.group_by('subject').agg(
    pl.len().alias('total_trial_count'),
    pl.col('velocity').n_unique().alias('velocity_count')
).sort('subject')
print(subject_total)

print("\n[4] 상세 Trial 목록 (Subject-Velocity-Trial-Step)")
print("-" * 50)
detailed = trial_level.sort(['subject', 'velocity', 'trial'])
print(detailed)

print("\n[5] 피험자별 Step/Nonstep Trial 개수 Summary")
print("-" * 50)
subject_summary = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count')
).sort(['subject', 'step_TF'])
print(subject_summary)

Current dir: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\analysis
CSV Path: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\output\all_trials_timeseries.csv
Exists: True

STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)

[1] Trial별 Step/Nonstep 분포
--------------------------------------------------
shape: (2, 3)
┌─────────┬─────────────┬───────────────┐
│ step_TF ┆ trial_count ┆ ratio_percent │
│ ---     ┆ ---         ┆ ---           │
│ str     ┆ u32         ┆ f64           │
╞═════════╪═════════════╪═══════════════╡
│ step    ┆ 53          ┆ 42.4          │
│ nonstep ┆ 72          ┆ 57.6          │
└─────────┴─────────────┴───────────────┘

[2] 피험자별 Step/Nonstep Trial 개수
--------------------------------------------------
shape: (45, 4)
┌─────────┬─────────┬─────────────┬─────────────────────┐
│ subject ┆ step_TF ┆ trial_count ┆ velocity_type_count │
│ ---     ┆ ---     ┆ ---         ┆ ---                 │
│ str     ┆ str     ┆ u32         ┆ u32                 │
╞══════

# 피험자별 평균 데이터

In [2]:
# 피험자별 평균값 계산 (주요 biomechanical 변수들)
numeric_cols = [
    'COM_X', 'COM_Y', 'COM_Z',
    'xCOM_X', 'xCOM_Y', 'xCOM_Z',
    'MOS_minDist_signed', 'MOS_AP_v3d', 'MOS_ML_v3d', 'MOS_v3d',
    'BOS_area',
    'KneeFlex_L_deg', 'KneeFlex_R_deg',
    'AnkleDorsi_L_deg', 'AnkleDorsi_R_deg',
    'GRF_X_N', 'GRF_Y_N', 'GRF_Z_N'
]

print("="*100)
print("피험자별 평균값 (Subject-level Aggregation)")
print("="*100)

subject_avg = df.select(['subject'] + numeric_cols).group_by('subject').agg(
    [pl.col(col).mean().round(4) for col in numeric_cols]
).sort('subject')

# COM 변수 표시
print("\n[1] COM (Center of Mass) - 무게중심")
print("-" * 80)
com_cols = subject_avg.select(['subject', 'COM_X', 'COM_Y', 'COM_Z'])
print(com_cols)

# xCOM 변수 표시
print("\n[2] xCOM (Extrapolated COM) - 외삽 무게중심")
print("-" * 80)
xcom_cols = subject_avg.select(['subject', 'xCOM_X', 'xCOM_Y', 'xCOM_Z'])
print(xcom_cols)

# MOS 변수 표시
print("\n[3] MOS (Margin of Stability) - 안정 여유도")
print("-" * 80)
mos_cols = subject_avg.select(['subject', 'MOS_minDist_signed', 'MOS_AP_v3d', 'MOS_ML_v3d', 'MOS_v3d'])
print(mos_cols)

# 기타 변수 표시
print("\n[4] BOS & Joint Angles - 지지면적 및 관절각")
print("-" * 80)
other_cols = subject_avg.select(['subject', 'BOS_area', 'KneeFlex_L_deg', 'KneeFlex_R_deg', 'AnkleDorsi_L_deg', 'AnkleDorsi_R_deg'])
print(other_cols)

# GRF 변수 표시
print("\n[5] GRF (Ground Reaction Force) - 지반반력")
print("-" * 80)
grf_cols = subject_avg.select(['subject', 'GRF_X_N', 'GRF_Y_N', 'GRF_Z_N'])
print(grf_cols)


피험자별 평균값 (Subject-level Aggregation)

[1] COM (Center of Mass) - 무게중심
--------------------------------------------------------------------------------
shape: (24, 4)
┌─────────┬─────────┬────────┬────────┐
│ subject ┆ COM_X   ┆ COM_Y  ┆ COM_Z  │
│ ---     ┆ ---     ┆ ---    ┆ ---    │
│ str     ┆ f64     ┆ f64    ┆ f64    │
╞═════════╪═════════╪════════╪════════╡
│ 가윤호  ┆ -1.4738 ┆ 0.4694 ┆ 1.1636 │
│ 강비은  ┆ -1.4833 ┆ 0.5086 ┆ 1.069  │
│ 권유영  ┆ -1.5081 ┆ 0.4889 ┆ 1.0838 │
│ 김민정  ┆ -1.4358 ┆ 0.4596 ┆ 1.0728 │
│ 김서하  ┆ -1.4664 ┆ 0.4882 ┆ 1.0739 │
│ 김우연  ┆ -1.448  ┆ 0.506  ┆ 1.0765 │
│ 김유민  ┆ -1.5429 ┆ 0.525  ┆ 1.1742 │
│ 김종철  ┆ -1.5189 ┆ 0.5068 ┆ 1.0907 │
│ 방수진  ┆ -1.5203 ┆ 0.5388 ┆ 1.1001 │
│ 방주원  ┆ -1.5076 ┆ 0.4844 ┆ 1.1063 │
│ 성효은  ┆ -1.5004 ┆ 0.4953 ┆ 1.1215 │
│ 안지연  ┆ -1.5177 ┆ 0.5204 ┆ 1.0964 │
│ 오나영  ┆ -1.5175 ┆ 0.4811 ┆ 1.0688 │
│ 유도경  ┆ -1.486  ┆ 0.4936 ┆ 1.0941 │
│ 유병한  ┆ -1.4765 ┆ 0.495  ┆ 1.1368 │
│ 유재원  ┆ -1.496  ┆ 0.4922 ┆ 1.1372 │
│ 이은수  ┆ -1.5606 ┆ 0.5192 ┆ 1.0741 │
│ 이인섭

# 피험자 전체 평균 데이터

In [3]:
print("\n" + "="*100)
print("전체 피험자 평균값 (Overall Mean)")
print("="*100)

overall_avg = df.select(numeric_cols).select(
    [pl.col(col).mean().round(4).alias(col) for col in numeric_cols]
)

print(overall_avg)

# 더 읽기 쉬운 형식으로 표시
print("\n" + "-"*100)
print("전체 평균값 (세로 포맷 - 변수명별)")
print("-"*100)

overall_T = overall_avg.melt(
    variable_name='Variable',
    value_name='Mean_Value'
).sort('Variable')

print(overall_T)


전체 피험자 평균값 (Overall Mean)
shape: (1, 18)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┐
│ COM ┆ COM ┆ COM ┆ xCO ┆ xCO ┆ xCO ┆ MOS ┆ MOS ┆ MOS ┆ MOS ┆ BOS ┆ Kne ┆ Kne ┆ AnkleD ┆ Ankle ┆ GRF_X ┆ GRF_Y ┆ GRF_Z │
│ _X  ┆ _Y  ┆ _Z  ┆ M_X ┆ M_Y ┆ M_Z ┆ _mi ┆ _AP ┆ _ML ┆ _v3 ┆ _ar ┆ eFl ┆ eFl ┆ orsi_L ┆ Dorsi ┆ _N    ┆ _N    ┆ _N    │
│ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ nDi ┆ _v3 ┆ _v3 ┆ d   ┆ ea  ┆ ex_ ┆ ex_ ┆ _deg   ┆ _R_de ┆ ---   ┆ ---   ┆ ---   │
│ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ st_ ┆ d   ┆ d   ┆ --- ┆ --- ┆ L_d ┆ R_d ┆ ---    ┆ g     ┆ f64   ┆ f64   ┆ f64   │
│     ┆     ┆     ┆     ┆     ┆     ┆ sig ┆ --- ┆ --- ┆ f64 ┆ f64 ┆ eg  ┆ eg  ┆ f64    ┆ ---   ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆ ned ┆ f64 ┆ f64 ┆     ┆     ┆ --- ┆ --- ┆        ┆ f64   ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆ --- ┆     ┆     ┆     ┆     ┆ f64 ┆ f64 ┆        ┆       ┆       ┆       

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_18088\2103004834.py:16: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  overall_T = overall_avg.melt(
